# Модуль 11. Разбор кейсов: QSPR растворимости ESOL

**Модуль 11. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** esol.csv: 1128 строк, измеренный logS; предрассчитанный ESOL-прогноз не используется.

Сравнить mean-baseline и Random Forest на двух групповых holdout без обучения преобразований на тесте.

**Workflow:** Паспорт → графовые признаки → identity/scaffold groups → train fit → test metrics → остатки.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab11_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%2011_QSAR.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [ ]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab11/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


## 1. Измеренная цель и группы

Количественная связь «структура–свойство» (QSPR) начинается с происхождения y. Используем измеренный logS, а готовый ESOL-прогноз исключаем из признаков и target. Одинаковая identity должна находиться только в одной части split. Каркасный holdout задаёт другую трудность переноса, поэтому его число строк может отличаться от 20%.

**Перед запуском:** Проверьте measured target и не включайте готовый predicted ESOL.

In [ ]:
from train_esol import load, run, TARGET
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
records=load(DATA/'esol.csv')
x=np.array([r['x'] for r in records]); y=np.array([r['y'] for r in records])
groups=[r['identity'] for r in records]
train,test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(x,y,groups))
assert not set(groups[i] for i in train)&set(groups[i] for i in test)
print('Target:',TARGET,'rows:',len(records),'train/test:',len(train),len(test))


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Измеренная цель и группы — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. fit только на train

Показываем последовательность группового split, Pipeline с imputer, mean baseline и Random Forest. Сначала выделяем train/test, затем fit получает только train. Даже вычисление медианы для заполнения пропусков на всём наборе перед split переносит информацию из test. Наблюдаемое преимущество RF относится к одному фиксированному эксперименту.

**Перед запуском:** Запустите фиксированные группы и Pipeline с imputer только на train.

In [ ]:
models={'baseline':make_pipeline(SimpleImputer(strategy='median'),DummyRegressor()),
        'RF':make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(
            n_estimators=200,min_samples_leaf=2,random_state=42,n_jobs=1))}
for name,model in models.items():
    model.fit(x[train],y[train])  # WHY: no fit/imputation on test.
    predicted=model.predict(x[test])
    print(name,'MAE logS:',mean_absolute_error(y[test],predicted))
# The full reference protocol also saves scaffold split, baseline and membership.
result=run(DATA/'esol.csv',OUT)
assert result['rows']==1128 and result['unique_identity']==1117
display(pd.json_normalize(result['splits'],sep='.'))


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. fit только на train — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Метрики и ошибки отдельных объектов

MAE даёт ошибку в единицах logS; RMSE сильнее реагирует на крупные отклонения. Сохраняем predictions и membership, а не только одно R². На parity-графике диагональ означает совпадение с измерением. Крупные остатки изучаем с ID и структурами, не удаляем их после просмотра, чтобы улучшить отчёт.

**Перед запуском:** Сравните MAE/RMSE/R² и ближайшую train-похожесть; изучите пять крупнейших остатков.

In [ ]:
predictions=pd.read_csv(OUT/'scaffold_predictions.csv')
predictions['absolute_error']=abs(predictions.measured_logS-predictions.predicted)
display(predictions.nlargest(5,'absolute_error'))
plt.scatter(predictions.measured_logS,predictions.predicted,alpha=.6)
plt.plot([-10,2],[-10,2],'k--');plt.xlabel('Measured logS');plt.ylabel('Predicted logS')
plt.savefig(OUT/'parity.png',dpi=150);plt.show()
artifact=result


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Метрики и ошибки отдельных объектов — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Identity:901/227, MAE RF≈0,497; scaffold:1009/119, MAE RF≈0,716 для зафиксированных зависимостей и seed. Эти значения не универсальная точность.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [ ]:
record={'module':11,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [scikit-learn: common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html).

Перейдите к [lab11_practice.ipynb](lab11_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.